# Part 1 - Neural Network Analysis

**Dataset:** Customer Churn Dataset  
**Goal:** Predict whether a customer will churn or not using a neural network

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
tf.random.set_seed(42)

---
## Task 1: Dataset Understanding

In [2]:
df = pd.read_csv('customer_churn_nn.csv')

print('Shape of dataset:', df.shape)
print('\nFirst 5 rows:')
df.head(3)

Shape of dataset: (2000, 17)

First 5 rows:


customer_id  region  plan_type contract_type payment_method  tenure_months  monthly_charges_inr  avg_login_days_per_month  support_tickets_last_90_days  payment_delay_days  data_usage_gb  satisfaction_score  last_complaint_days_ago  discount_percent  autopay_enabled  referral_count  churn
0    CUST0001   South   Standard  Month-to-month      Debit Card             30               687.40                        13                             0                   0          87.97                 8.0                       67                 0                0               0      0
1    CUST0002    West    Premium  Month-to-month          Wallet             15              1029.74                        22                             3                   1          82.17                 5.7                       69                 0                0               0      0
2    CUST0003  Central   Standard  Month-to-month     Credit Card             72               732.07                        13                             0                  11          89.39                 6.4                       63                10                0               0      0

In [3]:
print('Number of rows:', df.shape[0])
print('Number of columns:', df.shape[1])

print('\nColumn types:')
print(df.dtypes)

print('\nMissing values in each column:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())
print('Good, no missing values!')

Number of rows: 2000
Number of columns: 17

Column types:
customer_id                      object
region                           object
plan_type                        object
contract_type                    object
payment_method                   object
tenure_months                     int64
monthly_charges_inr             float64
avg_login_days_per_month          int64
support_tickets_last_90_days      int64
payment_delay_days                int64
data_usage_gb                   float64
satisfaction_score              float64
last_complaint_days_ago           int64
discount_percent                  int64
autopay_enabled                   int64
referral_count                    int64
churn                             int64
dtype: object

Missing values in each column:
customer_id                     0
region                          0
plan_type                       0
contract_type                   0
payment_method                  0
tenure_months                   0
monthly_char

In [4]:
print('Statistical summary of numerical columns:')
df.describe().T.head(12)

Statistical summary of numerical columns:


tenure_months  monthly_charges_inr  avg_login_days_per_month  support_tickets_last_90_days  payment_delay_days  data_usage_gb  satisfaction_score  last_complaint_days_ago  discount_percent  autopay_enabled  referral_count       churn
count    2000.000000          2000.000000               2000.000000                   2000.000000         2000.000000    2000.000000         2000.000000              2000.000000       2000.000000      2000.000000     2000.000000  2000.000000
mean       36.274500           849.701660                 15.491000                      1.497500            5.028000      99.938710            6.215050               100.218000          4.805000         0.501000        1.001000     0.015500
std        20.997308           288.977384                  5.754643                      1.705296            8.050990      49.742895            1.739975                57.719895          4.856217         0.500125        1.413277     0.123377

In [5]:
print('Target variable - churn distribution:')
print(df['churn'].value_counts())

churn_count = df['churn'].sum()
total = len(df)
print(f'\nChurn percentage: {churn_count/total*100:.2f}%')
print(f'Not Churn percentage: {(total-churn_count)/total*100:.2f}%')
print('\nThe data is quite imbalanced. Most customers did not churn.')

plt.figure(figsize=(5, 4))
df['churn'].value_counts().plot(kind='bar', color=['steelblue', 'salmon'])
plt.title('Churn Distribution')
plt.xlabel('Churn (0 = No, 1 = Yes)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

Target variable - churn distribution:
churn
0    1969
1      31
Name: count, dtype: int64

Churn percentage: 1.55%
Not Churn percentage: 98.45%

The data is quite imbalanced. Most customers did not churn.


In [6]:
cat_cols = ['region', 'plan_type', 'contract_type', 'payment_method']
print('Categorical columns:', cat_cols)
print('\nUnique values:')
for col in cat_cols:
    print(col, ':', list(df[col].unique()))

Categorical columns: ['region', 'plan_type', 'contract_type', 'payment_method']

Unique values:
region : ['South', 'West', 'Central', 'East', 'North']
plan_type : ['Standard', 'Premium', 'Basic']
contract_type : ['Month-to-month', 'Annual', 'Two-year']
payment_method : ['Debit Card', 'Wallet', 'Credit Card', 'Net Banking']


---
## Task 2: Data Preprocessing

In [7]:
# dropping customer id because its not a feature
df = df.drop('customer_id', axis=1)
print('Dropped customer_id column since its just an ID and not useful for training')

Dropped customer_id column since its just an ID and not useful for training


In [8]:
# encoding categorical columns
# using label encoder for all of them
print('Encoding categorical columns using LabelEncoder...')

le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print('Done encoding!')
print()
for col in cat_cols:
    print(col, ':', df[col].unique())

Encoding categorical columns using LabelEncoder...
Done encoding!

region : [0 1 2 3 4]
plan_type : [0 1 2]
contract_type : [0 1 2]
payment_method : [0 1 2 3]


In [9]:
X = df.drop('churn', axis=1)
y = df['churn']

print('X shape:', X.shape)
print('y shape:', y.shape)

# scaling features so that all values are on same scale
print('\nScaling the features using StandardScaler...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print('Done!')

X shape: (2000, 15)
y shape: (2000,)

Scaling the features using StandardScaler...
Done!


In [10]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print('Train set size:', X_train.shape[0])
print('Test set size:', X_test.shape[0])
print()
print('Churn in train:', y_train.sum(), 'out of', len(y_train))
print('Churn in test:', y_test.sum(), 'out of', len(y_test))

Train set size: 1600
Test set size: 400

Churn in train: 24 out of 1600
Churn in test: 7 out of 400


---
## Task 3: Neural Network Model Building

In [11]:
# building the neural network
# input layer -> hidden layer 1 -> hidden layer 2 -> output

model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 32)                512       
                                                                 
 dense_1 (Dense)             (None, 16)                528       
                                                                 
 dense_2 (Dense)             (None, 1)                 17        
                                                                 
Total params: 1057 (4.13 KB)
Trainable params: 1057 (4.13 KB)
Non-trainable params: 0 (0.00 B)
_________________________________________________________________


**Model design choices:**
- Input layer takes 15 features
- First hidden layer has 32 neurons with ReLU activation
- Second hidden layer has 16 neurons also with ReLU
- Output layer has 1 neuron with sigmoid because its binary classification (0 or 1)
- Loss function is binary crossentropy which is standard for binary classification
- Optimizer is Adam with default learning rate 0.001

---
## Task 4: Training and Evaluation

In [12]:
print('Training the model...')

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)

print('Training done!')

Training the model...
Epoch 1/30
45/45 [==============================] - 1s 8ms/step - loss: 0.3021 - accuracy: 0.9833 - val_loss: 0.2048 - val_accuracy: 0.9875
...
Epoch 30/30
45/45 [==============================] - 0s 2ms/step - loss: 0.0856 - accuracy: 0.9861 - val_loss: 0.1234 - val_accuracy: 0.9750
Training done!


In [13]:
test_loss, test_acc = model.evaluate(X_test, y_test)

print(f'\nTest Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')
print()
print(f'Training Accuracy (last epoch): {history.history["accuracy"][-1]:.4f}')
print(f'Validation Accuracy (last epoch): {history.history["val_accuracy"][-1]:.4f}')

13/13 [==============================] - 0s 1ms/step - loss: 0.0856 - accuracy: 0.9825

Test Loss: 0.0856
Test Accuracy: 0.9825

Training Accuracy (last epoch): 0.9861
Validation Accuracy (last epoch): 0.9750


In [14]:
y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten()

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:')
print(cm)
print()
print('Classification Report:')
print(classification_report(y_test, y_pred))

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Churn', 'Churn'],
            yticklabels=['Not Churn', 'Churn'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.tight_layout()
plt.show()

13/13 [==============================] - 0s 1ms/step

Confusion Matrix:
[[393   0]
 [  7   0]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      1.00      0.99       393
           1       0.00      0.00      0.00         7

    accuracy                           0.98       400
   macro avg       0.49      0.50      0.50       400
weighted avg       0.97      0.98      0.97       400



**Interpretation:**

The test accuracy is 98.25% which looks really good. The model is predicting most of the not-churn cases correctly (393 correct out of 393). 

But looking at the confusion matrix more carefully, the model predicted 0 (not churn) for all test samples. So all 7 churn cases were missed. This is probably happening because there are very few churn cases in the data (only 1.55%), so the model learned to just always predict 0.

The high accuracy is a bit misleading here. The recall for churn class is 0.00 which is bad. I should probably look at techniques like oversampling or class weights to fix this but that is beyond the current scope.

In [15]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linestyle='--')
ax1.set_title('Model Accuracy over Epochs')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train Loss', color='red')
ax2.plot(history.history['val_loss'], label='Val Loss', color='orange', linestyle='--')
ax2.set_title('Model Loss over Epochs')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.show()

---
## Task 5: Hyperparameter Experimentation

I ran 3 experiments changing different things each time.

In [16]:
# Experiment 1 is the baseline model we already built
print('=== Experiment 1: Baseline (already done above) ===')
print('2 hidden layers, neurons: 32 and 16, activation: relu')
print('Learning rate: 0.001, batch size: 32, epochs: 30')
print(f'Train Accuracy: {history.history["accuracy"][-1]:.4f}')
print(f'Val Accuracy: {history.history["val_accuracy"][-1]:.4f}')
print(f'Test Accuracy: {test_acc:.4f}')

=== Experiment 1: Baseline (already done above) ===
2 hidden layers, neurons: 32 and 16, activation: relu
Learning rate: 0.001, batch size: 32, epochs: 30
Train Accuracy: 0.9861
Val Accuracy: 0.9750
Test Accuracy: 0.9825


In [17]:
print('\n=== Experiment 2: More layers and neurons ===')
print('Training...')

model2 = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history2 = model2.fit(X_train, y_train, epochs=30, batch_size=32,
                      validation_split=0.1, verbose=0)

loss2, acc2 = model2.evaluate(X_test, y_test, verbose=0)
print(f'Train Accuracy: {history2.history["accuracy"][-1]:.4f}')
print(f'Val Accuracy: {history2.history["val_accuracy"][-1]:.4f}')
print(f'Test Accuracy: {acc2:.4f}')


=== Experiment 2: More layers and neurons ===
Training...
Train Accuracy: 0.9979
Val Accuracy: 0.9688
Test Accuracy: 0.9825


In [18]:
print('\n=== Experiment 3: Changed activation to tanh and higher learning rate ===')
print('Training...')

model3 = keras.Sequential([
    layers.Dense(32, activation='tanh', input_shape=(X_train.shape[1],)),
    layers.Dense(16, activation='tanh'),
    layers.Dense(1, activation='sigmoid')
])

model3.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.01),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history3 = model3.fit(X_train, y_train, epochs=30, batch_size=64,
                      validation_split=0.1, verbose=0)

loss3, acc3 = model3.evaluate(X_test, y_test, verbose=0)
print(f'Train Accuracy: {history3.history["accuracy"][-1]:.4f}')
print(f'Val Accuracy: {history3.history["val_accuracy"][-1]:.4f}')
print(f'Test Accuracy: {acc3:.4f}')


=== Experiment 3: Changed activation to tanh and higher learning rate ===
Training...
Train Accuracy: 0.9993
Val Accuracy: 0.9750
Test Accuracy: 0.9825


In [19]:
comparison_table = pd.DataFrame({
    'Experiment': ['Exp 1 (Baseline)', 'Exp 2 (More Layers)', 'Exp 3 (Tanh+High LR)'],
    'Hidden Layers': [2, 3, 2],
    'Neurons': ['32, 16', '64, 32, 16', '32, 16'],
    'Activation': ['ReLU', 'ReLU', 'Tanh'],
    'Learning Rate': [0.001, 0.001, 0.01],
    'Batch Size': [32, 32, 64],
    'Epochs': [30, 30, 30],
    'Train Accuracy': [0.9861, 0.9979, 0.9993],
    'Val Accuracy': [0.9750, 0.9688, 0.9750],
    'Test Accuracy': [0.9825, 0.9825, 0.9825]
})

print('Model Comparison Table:')
comparison_table

Model Comparison Table:


Experiment  Hidden Layers     Neurons Activation  Learning Rate  Batch Size  Epochs  Train Accuracy  Val Accuracy  Test Accuracy
0       Exp 1 (Baseline)              2      32, 16       ReLU          0.001          32      30          0.9861        0.9750         0.9825
1    Exp 2 (More Layers)              3  64, 32, 16       ReLU          0.001          32      30          0.9979        0.9688         0.9825
2  Exp 3 (Tanh+High LR)              2      32, 16       Tanh          0.010          64      30          0.9993        0.9750         0.9825

In [20]:
comparison_table.to_csv('results/model_comparison_table.csv', index=False)
print('Saved comparison table to results/model_comparison_table.csv')

**Observations from experiments:**

All three experiments got the exact same test accuracy of 98.25%. This is a bit strange but I think it's because all models are predicting 0 for every sample due to the class imbalance. So changing hyperparameters didn't really help.

One difference I noticed is that Experiment 2 (more layers) got a higher training accuracy (99.79%) compared to baseline (98.61%) which means it was fitting the training data better. But validation accuracy actually went down slightly (96.88% vs 97.50%). This could be a sign of slight overfitting.

Experiment 3 with tanh and higher learning rate got the highest train accuracy (99.93%) but same test accuracy. The higher learning rate seemed to make it converge faster on training data.

In [21]:
# generating final evaluation figure
# this has all 4 plots together

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle('Model Evaluation Outputs', fontsize=14, fontweight='bold')

# plot 1: accuracy
axes[0, 0].plot(history.history['accuracy'], label='Train Accuracy', color='blue')
axes[0, 0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange', linestyle='--')
axes[0, 0].set_title('Model 1 - Training Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# plot 2: loss
axes[0, 1].plot(history.history['loss'], label='Train Loss', color='red')
axes[0, 1].plot(history.history['val_loss'], label='Val Loss', color='purple', linestyle='--')
axes[0, 1].set_title('Model 1 - Training Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# plot 3: confusion matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1, 0],
            xticklabels=['Not Churn', 'Churn'],
            yticklabels=['Not Churn', 'Churn'])
axes[1, 0].set_title('Confusion Matrix - Model 1')
axes[1, 0].set_xlabel('Predicted')
axes[1, 0].set_ylabel('Actual')

# plot 4: test accuracy comparison
model_names = ['Exp 1\nBaseline', 'Exp 2\nMore Layers', 'Exp 3\nTanh+HighLR']
test_accs = [test_acc, acc2, acc3]
bars = axes[1, 1].bar(model_names, test_accs, color=['steelblue', 'coral', 'mediumseagreen'], width=0.5)
axes[1, 1].set_title('Test Accuracy Comparison')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_ylim(0.9, 1.01)
for bar, acc in zip(bars, test_accs):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                    f'{acc:.4f}', ha='center', va='bottom', fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('results/evaluation_outputs.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved to results/evaluation_outputs.png')

---
## Task 6: Final Reflection

### What role do weights and biases play in the model?

Weights and biases are the actual parameters that the neural network learns during training. Weights decide how much importance to give to each input feature. For example if satisfaction score is a strong indicator of churn, the weight connecting that feature to the neuron will become larger. Biases are like an offset - they help the neuron activate even when all inputs are zero. Without bias the neuron can only pass through the origin which limits what it can learn. Both weights and biases get updated in every training step using backpropagation.

### Why is an activation function required?

Without an activation function the neural network is basically just doing a linear transformation no matter how many layers you add. Multiple linear layers can always be collapsed into one linear layer, so you lose the advantage of having a deep network. Activation functions like ReLU add non-linearity to the model so it can learn complex patterns in the data. ReLU specifically just keeps positive values and sets negative ones to zero, which is simple but works well in practice.

### What happens when learning rate is too high or too low?

If learning rate is too high, the model updates its weights by large amounts in each step. This can cause the loss to jump around and not converge. Sometimes it can overshoot the minimum and actually get worse. If learning rate is too low, the model takes very tiny steps and training becomes very slow. It might also get stuck in a local minimum. The right learning rate is somewhere in between and 0.001 is generally a good starting point for Adam optimizer.

### Did your model show signs of underfitting or overfitting? Explain.

Looking at Experiment 2, the training accuracy was 99.79% but validation accuracy dropped to 96.88%. This gap between training and validation is a small sign of overfitting - the model is learning the training data too well and not generalizing as well to unseen data. The baseline model (Experiment 1) had a smaller gap so it is slightly better balanced.

Overall though I think the bigger issue is the class imbalance. All three models just predicted not-churn for everything which is why accuracy looks so high. The model is not really learning to identify churners. This is more of an underfitting problem for the minority class. To fix this I would need to use techniques like class weighting, oversampling the churn class, or using a different evaluation metric like F1 score or AUC-ROC instead of just accuracy.